# 14 · Reading and writing files (Spark)

Spark reads and writes many formats: CSV, JSON, **Parquet** (columnar, compressed, keeps types: the default
choice), ORC, Avro, and tables (Delta, Iceberg, BigQuery via a connector).

```
spark.read.format("csv").option("header", True).schema(...).load(path)
df.write.mode("overwrite").partitionBy("col").parquet(path)
```
A Spark "file" is really a **folder** with one part-file per partition. Practice files go to `tmp/` (not uploaded to git).

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/14_read_write_files.ipynb`](../../spark/practice/14_read_write_files.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. Read CSV: inferSchema vs explicit schema  ·  🟢 easy

Read `data/orders.tsv` (tab separated, header, NULL written as `NULL`) twice: (1) with `inferSchema=True`, (2) with an explicit schema. Print both schemas. What differs?

**How it works:** Read the same file twice; `printSchema` shows the guessed types vs the types you gave.

In [ ]:
path = str(ROOT / "data" / "orders.tsv")
inferred = (spark.read.option("header", True).option("sep", "\t")
            .option("nullValue", "NULL").option("inferSchema", True).csv(path))
inferred.printSchema()

explicit = (spark.read.option("header", True).option("sep", "\t").option("nullValue", "NULL")
            .schema("order_id INT, customer_id INT, order_date DATE, amount DECIMAL(10,2), status STRING")
            .csv(path))
explicit.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- status: string (nullable = true)

**Why / traps:** `inferSchema` guessed `amount` as **double** (a float: money can get rounding errors), not decimal(10,2). It also reads the file **one extra time** to guess. In production always give a schema.

## Q2. Bad rows: PERMISSIVE, DROPMALFORMED, FAILFAST  ·  🟡 medium

Write a small CSV with one bad row (`abc` in the amount column) to `tmp/bad.csv`. Read it with schema `id INT, amount DOUBLE` in the 3 modes and print the row count each time (FAILFAST will fail: catch the error). In PERMISSIVE mode keep the bad text in a `_corrupt` column.

**How it works:** PERMISSIVE keeps the row with NULLs and puts the raw line in `_corrupt`; DROPMALFORMED drops it; FAILFAST raises an error on read.

In [ ]:
bad = ROOT / "tmp" / "bad.csv"
bad.parent.mkdir(exist_ok=True)
bad.write_text("id,amount\n1,10.5\n2,abc\n3,7\n")

schema = "id INT, amount DOUBLE, _corrupt STRING"
permissive = (spark.read.option("header", True).option("mode", "PERMISSIVE")
              .option("columnNameOfCorruptRecord", "_corrupt").schema(schema).csv(str(bad)))
permissive.show()

dropped = spark.read.option("header", True).option("mode", "DROPMALFORMED") \
               .schema("id INT, amount DOUBLE").csv(str(bad))
print("DROPMALFORMED count():", dropped.count())          # 3 ?!
print("DROPMALFORMED real rows:", len(dropped.collect()))  # 2

try:
    spark.read.option("header", True).option("mode", "FAILFAST") \
         .schema("id INT, amount DOUBLE").csv(str(bad)).collect()
except Exception as e:
    print("FAILFAST error:", type(e).__name__)

+---+------+--------+
| id|amount|_corrupt|
+---+------+--------+
|  1|  10.5|    NULL|
|  2|  NULL|   2,abc|
|  3|   7.0|    NULL|
+---+------+--------+

DROPMALFORMED count(): 3
DROPMALFORMED real rows: 2
FAILFAST error: Py4JJavaError

**Why / traps:** Trap: `count()` does not parse the columns, so DROPMALFORMED does not see the bad row and says 3. `collect()` parses everything: 2. PERMISSIVE (default) = bad values become NULL, the raw line goes to the corrupt column: best for pipelines (count and route bad rows). DROPMALFORMED silently loses data. FAILFAST stops the job.

## Q3. Write and read Parquet  ·  🟢 easy

Write `orders` to `tmp/orders_pq` as Parquet (overwrite). Read it back, print the schema and the count. Then list the files in the folder.

**How it works:** `write.parquet` creates a folder of part-files; `read.parquet` gets the schema from the files.

In [ ]:
out = ROOT / "tmp" / "orders_pq"
orders.write.mode("overwrite").parquet(str(out))
back = spark.read.parquet(str(out))
back.printSchema()
print(back.count())
print(sorted(p.name for p in out.iterdir() if not p.name.startswith(".")))

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- status: string (nullable = true)

108
['_SUCCESS', 'part-00000-19538ce0-981d-4eb4-9267-73f9c1c15a07-c000.snappy.parquet']

**Why / traps:** Parquet keeps the schema inside the file, so no options are needed to read it. `_SUCCESS` is an empty marker file: the write finished.

## Q4. Save modes  ·  🟢 easy

Write `departments` to `tmp/dept_pq` with mode `overwrite`, then twice with `append`. Print the count after each. Then try mode `errorifexists` and catch the error, and `ignore`.

**How it works:** Each `append` adds the 7 rows again; `errorifexists` refuses an existing folder; `ignore` silently does nothing.

In [ ]:
out = str(ROOT / "tmp" / "dept_pq")
departments.write.mode("overwrite").parquet(out)
print("overwrite:", spark.read.parquet(out).count())
for i in range(2):
    departments.write.mode("append").parquet(out)
    print("append:", spark.read.parquet(out).count())
try:
    departments.write.mode("errorifexists").parquet(out)
except Exception as e:
    print("errorifexists ->", type(e).__name__)
departments.write.mode("ignore").parquet(out)     # does nothing, folder exists
print("after ignore:", spark.read.parquet(out).count())

overwrite: 7
append: 14
append: 21
errorifexists -> AnalysisException
after ignore: 21

**Why / traps:** `append` on a rerun = duplicates. Pipelines that may rerun should **overwrite a slice** (next question) or MERGE into a table format.

## Q5. Dynamic partition overwrite  ·  🔴 hard

Write orders partitioned by `order_month` (yyyy-MM text) to `tmp/orders_by_month`. Then rewrite **only** month `2025-10` with deduplicated rows, using `partitionOverwriteMode=dynamic`. Show that other months are untouched.

**How it works:** With dynamic mode, overwrite deletes and rewrites only the partition folders that appear in the new data.

In [ ]:
out = str(ROOT / "tmp" / "orders_by_month")
with_month = orders.withColumn("order_month", F.date_format("order_date", "yyyy-MM"))
with_month.write.mode("overwrite").partitionBy("order_month").parquet(out)

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
oct_clean = with_month.filter("order_month = '2025-10'").dropDuplicates()
oct_clean.write.mode("overwrite").partitionBy("order_month").parquet(out)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")    # back to default

back = spark.read.parquet(out)
back.groupBy("order_month").count().orderBy("order_month").filter("order_month >= '2025-08'").show()
print("total rows:", back.count())

+-----------+-----+
|order_month|count|
+-----------+-----+
|    2025-08|    5|
|    2025-09|    5|
|    2025-10|    4|
|    2025-11|    2|
|    2025-12|    6|
+-----------+-----+

total rows: 107

**Why / traps:** With the default (**static**) mode, `overwrite` deletes **all** partitions first: you would lose every other month. **Dynamic** mode replaces only the partitions present in the new data. This makes a daily job idempotent.

## Q6. Export a pipe-delimited CSV + control file  ·  🔴 hard

Like a real hand-off to another system: write **completed** orders (deduplicated) as **one** CSV file with a header, `|` as separator, dates as `yyyy-MM-dd`, to `tmp/export`. Then write a control file `tmp/export.ctl` with the row count and the sum of amount. Print both files' first lines.

**How it works:** `coalesce(1)` → one CSV part-file with `|` separator; then compute count and sum and write them to a small control file.

In [ ]:
out = ROOT / "tmp" / "export"
data = orders.filter("status = 'completed'").dropDuplicates()
(data.coalesce(1)                       # one output file (fine for small data)
     .write.mode("overwrite")
     .option("header", True).option("sep", "|")
     .option("dateFormat", "yyyy-MM-dd").option("nullValue", "")
     .csv(str(out)))

stats = data.agg(F.count("*").alias("n"), F.sum("amount").alias("total")).first()
(ROOT / "tmp" / "export.ctl").write_text(f"row_count|{stats.n}\namount_sum|{stats.total}\n")

part = next(out.glob("part-*.csv"))
print(part.read_text().splitlines()[:3])
print((ROOT / "tmp" / "export.ctl").read_text())

['order_id|customer_id|order_date|amount|status', '1014|27|2024-01-21|80.00|completed', '1021|20|2024-03-22|30.00|completed']
row_count|70
amount_sum|6115.00

**Why / traps:** `coalesce(1)` = one file, but also **one task** writes everything: OK for small exports, slow for big ones. The control file lets the receiver (e.g. an Oracle loader) **reconcile** count and sum before loading.

## Q7. JSON: write, read, multiline  ·  🟡 medium

Write `products` as JSON lines to `tmp/products_json` and read it back. Then read a **multi-line** JSON array file `tmp/one.json` (write it first) with `multiLine=True`.

**How it works:** JSON Lines is read by default; a JSON array or pretty file needs `multiLine=True`.

In [ ]:
out = str(ROOT / "tmp" / "products_json")
products.write.mode("overwrite").json(out)
spark.read.json(out).show(3)

(ROOT / "tmp" / "one.json").write_text('[\n {"id": 1, "tags": ["a", "b"]},\n {"id": 2, "tags": []}\n]')
spark.read.option("multiLine", True).json(str(ROOT / "tmp" / "one.json")).show()

+-----------+-----+----------+-----------------+
|   category|price|product_id|     product_name|
+-----------+-----+----------+-----------------+
|Electronics| 25.0|         1|   Wireless Mouse|
|Electronics| 12.0|         2|      USB-C Cable|
|Electronics| 60.0|         3|Bluetooth Speaker|
+-----------+-----+----------+-----------------+
only showing top 3 rows
+---+------+
| id|  tags|
+---+------+
|  1|[a, b]|
|  2|    []|
+---+------+

**Why / traps:** Spark expects **one JSON object per line** by default (JSON Lines). A pretty-printed file or a JSON array needs `multiLine=True`. Reading JSON without a schema scans the data to infer it.

## Q8. How many output files?  ·  🟢 easy

Write `order_items` with `repartition(3)` to `tmp/items_3` and with `coalesce(1)` to `tmp/items_1`. Count the part-files in each folder.

**How it works:** Each partition writes one part-file, so the partition count before writing = number of files.

In [ ]:
for name, df in [("items_3", order_items.repartition(3)), ("items_1", order_items.coalesce(1))]:
    out = ROOT / "tmp" / name
    df.write.mode("overwrite").parquet(str(out))
    print(name, len(list(out.glob("part-*"))), "part-files")

items_3 3 part-files
items_1 1 part-files

**Why / traps:** Number of files = number of partitions when writing (× partition folders with partitionBy). Too many small files is a classic performance problem; aim for files of ~128 MB–1 GB.

## Q9. Schema evolution: mergeSchema  ·  🟡 medium

Write 2 Parquet folders under `tmp/evolve`: `v=1` with columns (id, name), `v=2` with (id, name, city). Read the parent folder with and without `mergeSchema` and compare the columns.

**How it works:** `mergeSchema` reads the schemas of all files and combines them; without it Spark trusts one file.

In [ ]:
base = ROOT / "tmp" / "evolve"
customers.select("customer_id", "name").limit(3).write.mode("overwrite").parquet(str(base / "v=1"))
customers.select("customer_id", "name", "city").limit(3).write.mode("overwrite").parquet(str(base / "v=2"))
print(spark.read.parquet(str(base)).columns)
print(spark.read.option("mergeSchema", True).parquet(str(base)).columns)

['customer_id', 'name', 'v']
['customer_id', 'name', 'city', 'v']

**Why / traps:** Without mergeSchema Spark takes the schema from **one** file, so `city` can be missing. `v=1` / `v=2` folder names also become a column `v` (partition discovery).